# R12 · Preparación y diseño experimental sin fuga

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Cuaderno del diseño experimental de B, descrito en [`diseno_experimental.md`](diseno_experimental.md).

| Sección | Criterio de R12 |
|---|---|
| §1 Partición | Unidad de observación, de agrupación y partición; ningún bloque en dos conjuntos |
| §2 Variables | Predictoras con su fuente; excluidas con su motivo; prueba anti-fuga del historial |
| §3 Línea base y referencia | Línea base y modelo de referencia con métrica distinta de la exactitud |
| §4 Árbol | Profundidad elegida en validación, tamaño, brecha y una regla |

**La prueba no se usa aquí.** Todo se elige y se mide con entrenamiento y validación. El conjunto de
prueba se construye y se verifica que no comparte trimestres con los demás, pero ningún modelo se
evalúa con él: eso se hace una sola vez, al final del proyecto.

**Confidencialidad.** Solo se muestran cifras agregadas.

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import datos
from cartera import diseno as dis
from cartera import etiqueta as et
from cartera import historial as hi
from cartera.config import SEMILLA
from cartera.diseno import BLOQUE, CORTE, CORTE_VAL, FECHA_CORTE

np.random.seed(SEMILLA)

print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
saldos, movs = datos.leer_todo()
print("saldos:", saldos.shape, "| movimientos:", movs.shape)

saldos: (19262, 54) | movimientos: (40519, 44)


## 1. Partición

- **Unidad de observación:** la factura (`id_saldo`).
- **Unidad de agrupación:** el **trimestre de emisión**. No es el cliente: el modelo se
  usa con los mismos clientes, que se repiten mes a mes, y su información principal es su propio
  historial. Agrupar por cliente mediría el caso de clientes nuevos, que no es el uso real.
- **Partición temporal** (corte el 1/07/2025):

| Conjunto | Emisión | Regla de desenlace conocido |
|---|---|---|
| Subentrenamiento | 2022-01 a 2024-06 | `vencimiento + 30 ≤ 2024-07-01` |
| Validación | 2024-07 a 2025-06 | `vencimiento + 30 ≤ 2025-07-01` |
| Entrenamiento completo (subentrenamiento + validación) | 2022-01 a 2025-06 | `vencimiento + 30 ≤ 2025-07-01` |
| Prueba | 2025-07 a 2026-09 | `vencimiento + 30 ≤ 2026-10-18` |

La regla se aplica en cada corte: en cada conjunto solo entran facturas cuyo desenlace ya se conocía
en la fecha en que termina.

In [3]:
# Conjuntos con sus predictoras (src/cartera/diseno.py)
conj = dis.conjuntos(saldos, movs)
subent, val, ent, prueba = (conj[n] for n in ("subentrenamiento", "validación", "entrenamiento completo", "prueba"))

pd.DataFrame({n: {"facturas": len(d), "emisión desde": d.fecha.min().date(),
                  "emisión hasta": d.fecha.max().date(),
                  "trimestres": d.fecha.dt.to_period(BLOQUE).nunique()}
              for n, d in conj.items()}).T

,facturas,emisión desde,emisión hasta,trimestres
subentrenamiento,5972,2022-01-10,2024-05-30,10
validación,2661,2024-07-03,2025-05-30,4
entrenamiento completo,8930,2022-01-10,2025-05-30,14
prueba,3970,2025-07-01,2026-09-17,5


**Verificación: ningún trimestre ni ninguna factura queda en dos conjuntos.**

In [4]:
def trimestres(d):
    return set(d.fecha.dt.to_period(BLOQUE))

assert not trimestres(subent) & trimestres(val)
assert not trimestres(ent) & trimestres(prueba)
assert not set(subent.id_saldo) & set(val.id_saldo)
assert not set(ent.id_saldo) & set(prueba.id_saldo)
assert subent.fecha.max() < val.fecha.min() and ent.fecha.max() < prueba.fecha.min()
# La regla de desenlace conocido se cumple en cada corte
assert (subent.fecha_vencimiento + pd.Timedelta(days=30) <= CORTE_VAL).all()
assert (ent.fecha_vencimiento + pd.Timedelta(days=30) <= CORTE).all()
print("Sin trimestres ni facturas compartidas; regla de desenlace conocido cumplida en cada corte.")

Sin trimestres ni facturas compartidas; regla de desenlace conocido cumplida en cada corte.


**Lo que esta partición no evita del todo.** La etiqueta de una factura de entrenamiento usa su
cierre final. Si se cerró después del corte, saber cómo se cerró (con RC o con una nota) es información posterior. Su desenlace (tardía o no) ya estaba decidido en el corte, pero su
exclusión de la etiqueta puede depender de ese cierre. Se cuenta cuántas son:

In [5]:
cerro_despues = ent.fecha_cancelacion_registrada > CORTE
pd.Series({
    "Facturas de entrenamiento": len(ent),
    "Cerradas después del corte del 1/07/2025": int(cerro_despues.sum()),
    "% del entrenamiento": round(cerro_despues.mean() * 100, 2),
}, dtype=object).to_frame("valor")

,valor
Facturas de entrenamiento,8930
Cerradas después del corte del 1/07/2025,112
% del entrenamiento,1.25


## 2. Variables

**Predictoras.** Todas se conocen el día de emisión *t*. El grupo se trata
como una sola empresa: `empresa` no es predictora.

| Variable | Qué es | Fuente |
|---|---|---|
| `plazo_dias_observado` | Días entre emisión y vencimiento | Saldos |
| `log_monto_inicial` | Log del débito del primer movimiento | Movimientos (`src/cartera/muestra.py`) |
| `fecha_mes` | Mes de emisión | Saldos |
| `unidad_negocio`, `tipo_cliente` | Categóricas | Saldos |
| `atraso_prom_hist` | Atraso promedio de sus facturas canceladas antes de *t* | `src/cartera/historial.py` |
| `n_pagadas_hist` | Cuántas facturas canceló antes de *t* | `src/cartera/historial.py` |
| `saldo_abierto_t` | Saldo en facturas por sus movimientos anteriores a *t* | `src/cartera/historial.py` |
| `n_abiertas_t`, `n_vencidas_t` | Facturas abiertas y vencidas el día *t* | `src/cartera/historial.py` |
| `antiguedad_dias` | Días desde su primera factura | `src/cartera/historial.py` |

**Excluidas y su motivo.**

| Columnas | Motivo |
|---|---|
| `fecha_cancelacion_registrada`, `dias_hasta_cancelacion`, `total_creditos`, `saldo_neto_actual`, `porcentaje_pagado`, `num_recaudos_rc`, `num_creditos_no_rc`, `fecha_primer_recaudo`, `fecha_ultimo_recaudo`, `dias_hasta_primer_recaudo`, `tipo_documento_cierre`, `dias_mora_actual`, `estado_vencimiento_actual`, `num_movimientos` | No disponibles antes de la decisión (P); varias derivan del objetivo |
| «% de facturas con más de 30 días» y segmento de A | Derivadas del objetivo |
| Historial calculado con facturas posteriores a *t* | Calculado con el periodo que se predice; la prueba de abajo lo descarta |
| `total_debitos` | Acumulado que puede crecer después de la emisión; se usa el débito inicial |
| `codigo_cuenta` | Reclasificación posterior (difícil cobro) |
| `nit_cliente`, `cliente`, `id_*`, `numero_documento_origen` | Identificadores: el modelo memorizaría al cliente |
| `empresa` | El grupo se trata como una sola empresa; se usa solo para auditar errores por empresa |
| `nombre_vendedor` | Daría reglas por vendedor, el análisis descartado; solo para auditar errores |
| `condicion_pago_inferida` | Repite el plazo por tramos (R11 §4) |
| `actividad_economica_sintetica`, `cuota`, `fecha_radicacion` | Inventada, sin información y redundante (R11) |
| `fecha_vencimiento` y sus derivadas (`venc_*`), `fecha_anio`, `fecha_trimestre` | El vencimiento ya está en el plazo; el año y el trimestre son la unidad de agrupación |

**Atributos sensibles:** ninguno; la compañía anonimizó todo el conjunto.

In [6]:
NUM, CAT = dis.NUM, dis.CAT
print("Predictoras:", len(NUM) + len(CAT), "|", NUM + CAT)

Predictoras: 11 | ['plazo_dias_observado', 'log_monto_inicial', 'fecha_mes', 'atraso_prom_hist', 'n_pagadas_hist', 'saldo_abierto_t', 'n_abiertas_t', 'n_vencidas_t', 'antiguedad_dias', 'unidad_negocio', 'tipo_cliente']


**Prueba anti-fuga del historial.** Para 300 facturas al azar del entrenamiento, el historial
se recalcula de la forma más directa posible, filtrando fila por fila solo hechos con fecha anterior
a la emisión, y se compara con el valor de `historial.calcular`.

In [7]:
pagadas = et.facturas_cerradas(saldos)
pagadas = pagadas.assign(atraso=et.atraso_etiqueta(pagadas, movs)).dropna(subset=["atraso"])

revisadas, distintas = 0, 0
for i in ent.sample(300, random_state=SEMILLA).index:
    directo = hi.calcular_fuerza_bruta(ent.loc[i], saldos, movs, pagadas)
    for var, v in directo.items():
        a = ent.loc[i, var]
        revisadas += 1
        distintas += not ((np.isnan(a) and np.isnan(v)) or abs(a - v) < 1e-6)
assert distintas == 0
print(f"{revisadas} valores revisados en 300 facturas: {distintas} distintos.")

1800 valores revisados en 300 facturas: 0 distintos.


In [8]:
ent[hi.VARIABLES_HISTORIAL].describe().T[["count", "mean", "50%", "max"]].round(1)

,count,mean,50%,max
atraso_prom_hist,8457.0,30.1,22.6,358.8
n_pagadas_hist,8930.0,46.0,35.0,238.0
saldo_abierto_t,8930.0,8295906.8,3761412.2,209856810.7
n_abiertas_t,8930.0,2.3,1.0,24.0
n_vencidas_t,8930.0,1.8,1.0,23.0
antiguedad_dias,8930.0,804.5,787.5,1771.0


## 3. Línea base y modelo de referencia

Se entrena con el subentrenamiento y se mide en validación con **AUC-PR** (precisión promedio), que
no se infla con la clase minoritaria como la exactitud.

- **Línea base:** `DummyClassifier`, que predice siempre la proporción de tardías; su AUC-PR es esa
  proporción.
- **Regla simple sin modelo:** ordenar por `atraso_prom_hist` (sin historial, al final). Muestra
  cuánto se gana solo con el historial.
- **Modelo de referencia:** regresión logística con más peso a las tardías (`class_weight`), con
  imputación, escala y codificación dentro del `Pipeline`. El faltante de `atraso_prom_hist`
  (cliente sin historial) se imputa con la mediana del entrenamiento y se marca con una columna
  indicadora.

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline

X_sub, y_sub = subent[NUM + CAT], subent.tardia
X_val, y_val = val[NUM + CAT], val.tardia

def metricas(nombre, y, puntaje):
    return {"modelo": nombre, "AUC-PR": round(average_precision_score(y, puntaje), 3),
            "AUC-ROC": round(roc_auc_score(y, puntaje), 3)}

prep_lineal = dis.prep_lineal()
logistica = Pipeline([("prep", prep_lineal),
                      ("est", LogisticRegression(class_weight="balanced", max_iter=2000))]).fit(X_sub, y_sub)
dummy = DummyClassifier(strategy="prior").fit(X_sub, y_sub)

filas = [
    metricas("Línea base (DummyClassifier)", y_val, dummy.predict_proba(X_val)[:, 1]),
    metricas("Regla simple: atraso promedio histórico", y_val, X_val.atraso_prom_hist.fillna(-1e9)),
    metricas("Regresión logística", y_val, logistica.predict_proba(X_val)[:, 1]),
]
pd.DataFrame({"subentrenamiento": [len(subent), round(y_sub.mean() * 100, 1)],
              "validación": [len(val), round(y_val.mean() * 100, 1)]},
             index=["facturas", "% tardías"])

,subentrenamiento,validación
facturas,5972.0,2661.0
% tardías,30.2,30.9


In [10]:
referencia = pd.DataFrame(filas).set_index("modelo")
referencia

,AUC-PR,AUC-ROC
modelo,,
Línea base (DummyClassifier),0.309,0.500
Regla simple: atraso promedio histórico,0.673,0.799
Regresión logística,0.724,0.844


## 4. Árbol de decisión

**Cómo se elige la poda.** Se prueban profundidades máximas de 2 a 10 y sin límite, y mínimos de 20,
50 y 100 facturas por hoja, con `class_weight = "balanced"`. Todo se ajusta con el
subentrenamiento y se compara en validación. **Regla de elección:** el árbol con **menos hojas**
cuya AUC-PR en validación está a menos de 0,01 de la mejor; un árbol más grande que no mejora de
forma distinguible solo es más difícil de leer.

El árbol no necesita escalar, y trata el faltante de `atraso_prom_hist` (cliente sin historial)
como una rama propia.

In [11]:
from sklearn.tree import DecisionTreeClassifier, export_text

prep_arbol = dis.prep_arbol()

def arbol(profundidad, min_hoja):
    est = DecisionTreeClassifier(max_depth=profundidad, min_samples_leaf=min_hoja,
                                 class_weight="balanced", random_state=SEMILLA)
    return Pipeline([("prep", prep_arbol), ("est", est)])

filas = []
for prof in [2, 3, 4, 5, 6, 8, 10, None]:
    for min_hoja in [20, 50, 100]:
        a = arbol(prof, min_hoja).fit(X_sub, y_sub)
        filas.append({"profundidad máx.": "sin límite" if prof is None else prof, "mín. por hoja": min_hoja,
                      "hojas": a.named_steps["est"].get_n_leaves(),
                      "AUC-PR subentr.": average_precision_score(y_sub, a.predict_proba(X_sub)[:, 1]),
                      "AUC-PR validación": average_precision_score(y_val, a.predict_proba(X_val)[:, 1])})
rejilla = pd.DataFrame(filas)
mejor = rejilla["AUC-PR validación"].max()
elegible = rejilla[rejilla["AUC-PR validación"] >= mejor - 0.01].sort_values(["hojas", "AUC-PR validación"],
                                                                             ascending=[True, False])
elegida = elegible.iloc[0]
print(f"Mejor AUC-PR en validación: {mejor:.3f}")
rejilla.round(3)

Mejor AUC-PR en validación: 0.673


,profundidad máx.,mín. por hoja,hojas,AUC-PR subentr.,AUC-PR validación
0,2,20,4,0.506,0.533
1,2,50,4,0.506,0.533
2,2,100,4,0.506,0.533
3,3,20,8,0.567,0.610
4,3,50,8,0.567,0.610
5,3,100,7,0.567,0.605
6,4,20,14,0.598,0.624
7,4,50,14,0.598,0.624
8,4,100,12,0.609,0.632
9,5,20,27,0.645,0.653


In [12]:
prof = None if elegida["profundidad máx."] == "sin límite" else int(elegida["profundidad máx."])
arbol_ref = arbol(prof, int(elegida["mín. por hoja"])).fit(X_sub, y_sub)
est = arbol_ref.named_steps["est"]
ap_sub = average_precision_score(y_sub, arbol_ref.predict_proba(X_sub)[:, 1])
ap_val = average_precision_score(y_val, arbol_ref.predict_proba(X_val)[:, 1])
pd.Series({
    "Profundidad máxima elegida": elegida["profundidad máx."],
    "Mínimo de facturas por hoja": int(elegida["mín. por hoja"]),
    "Profundidad real": est.get_depth(),
    "Hojas": est.get_n_leaves(),
    "AUC-PR subentrenamiento": round(ap_sub, 3),
    "AUC-PR validación": round(ap_val, 3),
    "Brecha (subentrenamiento − validación)": round(ap_sub - ap_val, 3),
}, dtype=object).to_frame("árbol de referencia")

,árbol de referencia
Profundidad máxima elegida,6
Mínimo de facturas por hoja,50
Profundidad real,6
Hojas,36
AUC-PR subentrenamiento,0.679
AUC-PR validación,0.668
Brecha (subentrenamiento − validación),0.011


**Brecha.** R12 pide la brecha entre entrenamiento y prueba. Aquí se reporta la de
subentrenamiento frente a validación, porque la prueba se consulta una sola vez, al final del
proyecto (`diseno_experimental.md` §6.6). Esta interpretación está en consulta con el docente (`PENDIENTES.md` ›
Dudas para el docente).

**Reglas del árbol.** Cada camino de la raíz a una hoja es una regla; el número final es la clase
que predice (1 = tardía). Las reglas describen lo que usa el modelo, no lo que causa el atraso.

In [13]:
print(export_text(est, feature_names=list(arbol_ref.named_steps["prep"].get_feature_names_out()),
                  max_depth=4, decimals=1))

|--- atraso_prom_hist <= 32.8
|   |--- n_vencidas_t <= 0.5
|   |   |--- n_pagadas_hist <= 4.5
|   |   |   |--- unidad_negocio_Unidad 01 <= 0.5
|   |   |   |   |--- atraso_prom_hist <= 1.8
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- atraso_prom_hist >  1.8
|   |   |   |   |   |--- class: 1
|   |   |   |--- unidad_negocio_Unidad 01 >  0.5
|   |   |   |   |--- class: 0
|   |   |--- n_pagadas_hist >  4.5
|   |   |   |--- atraso_prom_hist <= 10.7
|   |   |   |   |--- unidad_negocio_Unidad 02 <= 0.5
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |   |--- unidad_negocio_Unidad 02 >  0.5
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |--- atraso_prom_hist >  10.7
|   |   |   |   |--- n_pagadas_hist <= 22.5
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |   |   |--- n_pagadas_hist >  22.5
|   |   |   |   |   |--- truncated branch of depth 2
|   |--- n_vencidas_t >  0.5
|   |   |--- atraso_prom_hist <= 10.9
|   |   |   |--- n_pagadas_h

In [14]:
importancia = pd.Series(est.feature_importances_,
                        index=arbol_ref.named_steps["prep"].get_feature_names_out())
importancia[importancia > 0].sort_values(ascending=False).round(3).to_frame("importancia (Gini)")

,importancia (Gini)
atraso_prom_hist,0.564
n_vencidas_t,0.185
n_pagadas_hist,0.152
saldo_abierto_t,0.037
plazo_dias_observado,0.035
antiguedad_dias,0.009
fecha_mes,0.007
unidad_negocio_Unidad 01,0.007
log_monto_inicial,0.003
unidad_negocio_Unidad 02,0.001


In [15]:
pd.concat([referencia, pd.DataFrame([metricas("Árbol de referencia", y_val, arbol_ref.predict_proba(X_val)[:, 1])]).set_index("modelo")])

,AUC-PR,AUC-ROC
modelo,,
Línea base (DummyClassifier),0.309,0.500
Regla simple: atraso promedio histórico,0.673,0.799
Regresión logística,0.724,0.844
Árbol de referencia,0.668,0.810
